In [2]:
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as Sampler
from qiskit_ibm_runtime.sampler import SamplerOptions
from qiskit_ibm_runtime.options.sampler_execution_options import SamplerExecutionOptionsV2
service = QiskitRuntimeService()
backend = service.backend("ibm_torino", use_fractional_gates=False)
opts = SamplerOptions(
default_shots=1024,
execution=SamplerExecutionOptionsV2(meas_type="classified"),
)
sampler = Sampler(backend, options=opts)

In [3]:
from qiskit import QuantumCircuit

def zz_evolve(qc: QuantumCircuit, q0: int, q1: int, theta: float):
    # e^{-i theta Z⊗Z} via CZ-RZ-CZ; geeft alleen RZ/CZ
    qc.cz(q0, q1)
    qc.rz(2*theta, q1)   # of op q0, equivalent tot globale fase
    qc.cz(q0, q1)

def xx_evolve(qc: QuantumCircuit, q0: int, q1: int, theta: float):
    qc.h(q0); qc.h(q1)
    zz_evolve(qc, q0, q1, theta)
    qc.h(q0); qc.h(q1)

def yy_evolve(qc: QuantumCircuit, q0: int, q1: int, theta: float):
    qc.sdg(q0); qc.h(q0)
    qc.sdg(q1); qc.h(q1)
    zz_evolve(qc, q0, q1, theta)
    qc.h(q0); qc.s(q0)
    qc.h(q1); qc.s(q1)


In [4]:
import numpy as np
from qiskit import QuantumCircuit

def build_sqdrift_circuit(n_qubits, excitations, coeffs, t, k, N_steps, initial_layout=None):
    """
    excitations: lijst van geselecteerde fermion-termen voor dit circuit (b.v. 10, 15 of 25 stuks)
                 Na JW-map: elk element een mini-‘gate recipe’ voor 1- of 2-body evolutie op (p,q,...) 
                 waarbij je uiteindelijk op {X⊗X, Y⊗Y, Z⊗Z} decompositie uitkomt per support.
    coeffs: |c_i| voor kansverdeling; sum(coeffs) = λ
    """
    qc = QuantumCircuit(n_qubits)
    # Startstate: HF-determinant (π-occupaties); laad die als bitstring, bijvoorbeeld via qc.x(...)
    # (Laat HF-berekening dit bitstring aanleveren)
    # ...
    lam = float(np.sum(np.abs(coeffs)))
    probs = np.abs(coeffs) / lam
    theta_unit = lam * (t * k) / N_steps  # qDRIFT hoek per micro-stap

    for _ in range(N_steps):
        j = np.random.choice(len(excitations), p=probs)
        term = excitations[j]    # bevat info of het XX/YY/ZZ-achtig is op welke qubits en het teken
        sign = np.sign(term["coeff"])
        angle = sign * theta_unit

        # Voorbeeld: term["type"] in {"XX","YY","ZZ","Z","X","Y"} en term["qubits"] = (q0,q1) of (q,)
        if term["type"] == "ZZ":
            q0, q1 = term["qubits"]; zz_evolve(qc, q0, q1, angle)
        elif term["type"] == "XX":
            q0, q1 = term["qubits"]; xx_evolve(qc, q0, q1, angle)
        elif term["type"] == "YY":
            q0, q1 = term["qubits"]; yy_evolve(qc, q0, q1, angle)
        elif term["type"] == "Z":
            (q,) = term["qubits"]; qc.rz(2*angle, q)
        elif term["type"] == "X":
            (q,) = term["qubits"]; qc.rx(2*angle, q)  # RX is basispoort op Torino
        elif term["type"] == "Y":
            (q,) = term["qubits"]; qc.ry(2*angle, q)  # wordt gedecomposeerd; ok
        # (Andere gevallen: eerst naar {X,Y,Z}-canonische vorm roteren, dan terug)
    qc.barrier()
    qc.measure_all()
    return qc


In [5]:
from qiskit import transpile

# Minimal set van 10 termen voor qDRIFT (1-body Z-termen op qubits 0..9)
selected_fermion_terms_10 = [{"type": "Z", "qubits": (q,), "coeff": 1.0} for q in range(10)]

qc = build_sqdrift_circuit(
    n_qubits=48,
    excitations=selected_fermion_terms_10,  # 10 per circuit voor ondiepe batch
    coeffs=np.abs([t["coeff"] for t in selected_fermion_terms_10]),
    t=1.0, k=1, N_steps=50
)

qc_t = transpile(
    qc, backend,
    optimization_level=3,
    routing_method="sabre"
)

# Gebruik de bestaande sampler uit cel 1
job = sampler.run([qc_t])
res = job.result()
pub = res[0] # SamplerPubResult
bits = pub.join_data() # BitArray (meas_type='classified')

counts = bits.get_counts() # dict[str, int]
bitstrings = bits.get_bitstrings() # list[str] (length = shots)
# Maak een lijst van bitstrings met multipliciteit (lengte ~ shots)



In [6]:
# We bouwen/gebruik(en) de subruimte 'bs' en diagona liseren de projectie van H_qubit.
from qiskit_addon_sqd.qubit import sort_and_remove_duplicates, project_operator_to_subspace
k = 1  # ground state
import numpy as np
from qiskit.quantum_info import SparsePauliOp
from qiskit_addon_sqd.qubit import sort_and_remove_duplicates, project_operator_to_subspace
H_qubit = SparsePauliOp.from_list([("Z", 1.0)])

# Gebruik bestaande 'bs' als beschikbaar; anders construeer een minimale subruimte.
if 'bs' in globals():
	bs_local = bs
else:
	# Fallback: neem 1-qubit samples uit beschikbare variabelen.
	if 'bitstring_list' in globals():
		_mat = np.array([[int(b) for b in s] for s in bitstring_list], dtype=bool)
	elif 'bitstrings' in globals():
		# Reduceer 48-bit strings naar 1 qubit (bijv. laatste bit)
		_mat = np.array([[int(s[-1])] for s in bitstrings], dtype=bool)
	else:
		_mat = np.array([[0], [1]], dtype=bool)
	bs_local = sort_and_remove_duplicates(_mat)

d = bs_local.shape[0]
Hproj = project_operator_to_subspace(bs_local, H_qubit)

# Voor kleine d fallback naar dense eignwaarde-oplossing
if k >= d - 1:
	w, v = np.linalg.eigh(Hproj.toarray())  # Hermitisch
	energy, eigvec = w[:k][0], v[:, :k][:, 0]
else:
	E_s, V_s = solve_qubit(bs_local, H_qubit, k=k, which="SA")
	energy, eigvec = E_s[0], V_s[:, 0]

print("Energy:", energy)
print("Eigenvector (subspace basis):", eigvec)


Energy: -1.0
Eigenvector (subspace basis): [0.+0.j 1.+0.j]


In [ ]:
qc.draw("mpl")

In [4]:
# qdrift_z0_plus_z1.py
from __future__ import annotations
import numpy as np
from qiskit import QuantumCircuit, transpile
from qiskit.transpiler import CouplingMap
# from qiskit_ibm_runtime import QiskitRuntimeService, Sampler  # voor echt hardwaregebruik

# ============
# Parameters
# ============
t          = 0.8          # evolutietijd (in 'H-units')
N_steps    = 50           # qDRIFT micro-steps per circuit
N_random   = 5            # aantal qDRIFT-randomisaties om te bouwen
shots      = 1024         # shots voor metingen
seed       = 12345        # RNG seed (reproduceerbaarheid)
use_h_sandwich = True    # zet True om H-sandwich te gebruiken (effectief X-evolutie)

rng = np.random.default_rng(seed)

# =====================================================
# Exacte evolutie voor H = Z0 + Z1  (commuteren → kort)
# =====================================================
def exact_circuit_z0_plus_z1(t: float, measure: bool = True, use_h: bool = False) -> QuantumCircuit:
    """
    Exact U(t) = e^{-i t (Z0 + Z1)}.
    Zonder globale fase geldt: e^{-i t Z} = RZ(2t).
    Optioneel H-sandwich voor effectieve X-evolutie (H Z H = X).
    """
    qc = QuantumCircuit(2, 2 if measure else 0)
    if use_h:
        qc.h([0, 1])
    qc.rz(2*t, 0)
    qc.rz(2*t, 1)
    if use_h:
        qc.h([0, 1])
    if measure:
        qc.measure_all()
    return qc

# ========================================================
# qDRIFT voor H = Z0 + Z1  (lambda = |1| + |1| = 2)
# ========================================================
def qdrift_circuit_z0_plus_z1(t: float,
                              N_steps: int,
                              rng: np.random.Generator,
                              measure: bool = True,
                              use_h: bool = False) -> QuantumCircuit:
    """
    qDRIFT: bij elke micro-stap kies je Z0 of Z1 met kans 1/2,
    en pas toe: exp(-i * (lambda * t / N) * Zk).
    Omdat RZ(2θ) = e^{-i θ Z} (tot globale fase), is de gate:
        RZ( 2 * lambda * t / N )  =  RZ( 4 t / N )
    op de gekozen qubit k (0 of 1).
    """
    lam = 2.0
    theta = lam * t / N_steps   # hoek in exponent
    rz_angle = 2 * theta    # 2 * (lam t / N) * 2 = 4 t / N

    qc = QuantumCircuit(2, 2 if measure else 0)
    if use_h:
        qc.h([0, 1])  # H-sandwich IN
    for _ in range(N_steps):
        k = rng.integers(0, 2)    # 0 of 1
        qc.rz(rz_angle, k)
    if use_h:
        qc.h([0, 1])  # H-sandwich UIT
    if measure:
        qc.measure_all()
    return qc

# =====================================================
# Transpile (poorten/basis en topologie richting ibm_torino)
# =====================================================
# Basisgates van ibm_torino zijn o.a.: {'rz','sx','x','cz','id'} (en meet)
basis_gates = ['rz', 'sx', 'x', 'cz', 'id', 'measure']

# Een simpele 2-qubit koppeling (lineaire keten 1-0) werkt voor ons toy:
coupling = CouplingMap(couplinglist=[[0,1],[1,0]])

def transpile_for_torino(qc: QuantumCircuit) -> QuantumCircuit:
    """
    Transpileer naar een set die met ibm_torino compatible is:
    - basis_gates passend bij Torino
    - eenvoudige 2-qubit koppeling
    - optimalisatie op max
    """
    tqc = transpile(
        qc,
        basis_gates=basis_gates,
        coupling_map=coupling,
        optimization_level=3,
        seed_transpiler=seed
    )
    return tqc

# ===========
# Build demo
# ===========
if __name__ == "__main__":
    print("=== Exact circuit (H = Z0 + Z1) ===")
    qc_exact = exact_circuit_z0_plus_z1(t, measure=True, use_h=use_h_sandwich)
    tqc_exact = transpile_for_torino(qc_exact)
    print(tqc_exact)
    print(f"depth={tqc_exact.depth()}, 2q_count={tqc_exact.num_connected_components()} (niet betekenisvol), size={tqc_exact.size()}")

    print("\n=== qDRIFT circuits (randomizations) ===")
    qdrift_transpiled = []
    for i in range(N_random):
        qc_qd = qdrift_circuit_z0_plus_z1(t, N_steps, rng, measure=True, use_h=use_h_sandwich)
        tqc_qd = transpile_for_torino(qc_qd)
        qdrift_transpiled.append(tqc_qd)
        print(f"\n-- Randomization {i+1} --")
        print(tqc_qd)
        print(f"depth={tqc_qd.depth()}, size={tqc_qd.size()}")

    # ================================
    # (Optioneel) lokaal simuleren
    # ================================
    try:
        from qiskit_aer import Aer
        backend = Aer.get_backend("aer_simulator")
        print("\nSimulating exact circuit...")
        res_exact = backend.run(tqc_exact, shots=shots).result()
        print("Counts (exact):", res_exact.get_counts())

        for i, tqc_qd in enumerate(qdrift_transpiled, 1):
            print(f"\nSimulating qDRIFT randomization {i}...")
            res_qd = backend.run(tqc_qd, shots=shots).result()
            print(f"Counts (qDRIFT {i}):", res_qd.get_counts())
    except Exception as e:
        print("\nAer simulator niet beschikbaar; transpile output is gebouwd. Fout:", e)

    # =========================================================
    # (Optioneel) IBM Runtime – hardware job op ibm_torino
    # (commented out; activeer als je Runtime geconfigureerd hebt)
    # =========================================================
    """
    service = QiskitRuntimeService()
    backend = service.backend("ibm_torino", use_fractional_gates=False)
    sampler = Sampler(backend=backend, options={"execution": {"shots": shots}, "resilience_level": 1})
    # Exact
    job = sampler.run([tqc_exact])
    print("Exact (hardware) quasi_dists:", job.result().quasi_dists)
    # qDRIFT batch
    job = sampler.run(qdrift_transpiled)
    print("qDRIFT (hardware) quasi_dists:", job.result().quasi_dists)
    """


=== Exact circuit (H = Z0 + Z1) ===
global phase: π
         ┌─────────┐┌────┐┌─────────────┐┌────┐┌─────────┐ ░ ┌─┐   
q_0 -> 0 ┤ Rz(π/2) ├┤ √X ├┤ Rz(-1.5416) ├┤ √X ├┤ Rz(π/2) ├─░─┤M├───
         ├─────────┤├────┤├─────────────┤├────┤├─────────┤ ░ └╥┘┌─┐
q_1 -> 1 ┤ Rz(π/2) ├┤ √X ├┤ Rz(-1.5416) ├┤ √X ├┤ Rz(π/2) ├─░──╫─┤M├
         └─────────┘└────┘└─────────────┘└────┘└─────────┘ ░  ║ └╥┘
 meas: 2/═════════════════════════════════════════════════════╩══╩═
                                                              0  1 
depth=6, 2q_count=4 (niet betekenisvol), size=12

=== qDRIFT circuits (randomizations) ===

-- Randomization 1 --
global phase: π
         ┌─────────┐┌────┐┌─────────────┐┌────┐┌─────────┐ ░ ┌─┐   
q_0 -> 0 ┤ Rz(π/2) ├┤ √X ├┤ Rz(-1.4136) ├┤ √X ├┤ Rz(π/2) ├─░─┤M├───
         ├─────────┤├────┤├─────────────┤├────┤├─────────┤ ░ └╥┘┌─┐
q_1 -> 1 ┤ Rz(π/2) ├┤ √X ├┤ Rz(-1.6696) ├┤ √X ├┤ Rz(π/2) ├─░──╫─┤M├
         └─────────┘└────┘└─────────────┘└────┘└─────────┘ ░  ║ └╥┘
 m

In [5]:
from collections import Counter

def _split_key(k: str) -> str:
    """Haal de linker bitstring (qubits) uit keys als '11 00' of '0011'."""
    k = k.strip()
    if " " in k:
        # neem links deel (qubit register) en strip spaties
        k = k.split()[0]
    return k

def expectation_Zs_from_counts(counts: dict) -> dict:
    """
    counts: dict zoals {'00': 242, '01': 209, ...} of {'01 00': 326, ...}
    Retourneert <Z0>, <Z1>, <H>.
    Conventie: bitstring[0] = MSB (qubit 0 links in je diagram).
    """
    # normaliseer keys en totalen
    flat = Counter()
    total = 0
    for k, v in counts.items():
        key = _split_key(k)
        flat[key] += v
        total += v
    if total == 0:
        return {"Z0": 0.0, "Z1": 0.0, "H": 0.0}

    # kansen
    P = {k: flat[k] / total for k in flat}

    # marges: P(0*) = P(00)+P(01), P(1*) = P(10)+P(11)
    P00 = P.get("00", 0.0)
    P01 = P.get("01", 0.0)
    P10 = P.get("10", 0.0)
    P11 = P.get("11", 0.0)

    P0star = P00 + P01     # qubit0 = 0
    P1star = P10 + P11     # qubit0 = 1
    Pstar0 = P00 + P10     # qubit1 = 0
    Pstar1 = P01 + P11     # qubit1 = 1

    # <Z> = P(0) - P(1)
    Z0 = P0star - P1star
    Z1 = Pstar0 - Pstar1
    H  = Z0 + Z1
    return {"Z0": Z0, "Z1": Z1, "H": H}

# ---- Voorbeeldgebruik met jouw print-outputs ----
exact_counts = {'00 00': 247, '11 00': 267, '10 00': 277, '01 00': 233}
qd1_counts   = {'01 00': 326, '11 00': 268, '10 00': 197, '00 00': 233}

print("Exact  :", expectation_Zs_from_counts(exact_counts))
print("qDRIFT1:", expectation_Zs_from_counts(qd1_counts))

# ---- Gemiddelde over meerdere qDRIFT randomizations ----
def average_expectations(list_of_counts):
    vals = [expectation_Zs_from_counts(c) for c in list_of_counts]
    n = len(vals) or 1
    Z0 = sum(v["Z0"] for v in vals)/n
    Z1 = sum(v["Z1"] for v in vals)/n
    H  = sum(v["H"]  for v in vals)/n
    return {"Z0": Z0, "Z1": Z1, "H": H}

qd_counts_list = [
    {'01 00': 326, '11 00': 268, '10 00': 197, '00 00': 233},
    {'01 00': 209, '11 00': 256, '10 00': 317, '00 00': 242},
    {'00 00': 204, '11 00': 242, '10 00': 145, '01 00': 433},
    {'01 00': 274, '00 00': 241, '11 00': 289, '10 00': 220},
    {'00 00': 198, '01 00': 471, '10 00': 107, '11 00': 248},
]
print("qDRIFT avg:", average_expectations(qd_counts_list))


Exact  : {'Z0': -0.0625, 'Z1': 0.0234375, 'H': -0.0390625}
qDRIFT1: {'Z0': 0.091796875, 'Z1': -0.16015625, 'H': -0.068359375}
qDRIFT avg: {'Z0': 0.105859375, 'Z1': -0.178125, 'H': -0.072265625}
